# QWEN-999B: Core Training Pipeline on Google Colab / Kaggle (T4 15 GB)
**Objective:** Train QWEN-999B using Unsloth 4-bit QLoRA on a free T4 GPU (15 GB VRAM) with Zero-OOM guarantee.

### Guardrails Applied:
- **VRAM Constraint:** `fp16=True`, `bf16=False` (T4 lacks native BF16 arithmetic support)
- **Disconnection Resilience:** `save_strategy='steps'`, `save_steps=50`, `resume_from_checkpoint=True`, and auto-push to Hugging Face
- **Repo Hygiene:** Hugging Face token read from Colab Secrets / Environment variable (`HF_TOKEN`), never hardcoded

### 1. Environment Setup & Dependency Installation
Install Unsloth, PyTorch, Transformers, and TRL optimized for NVIDIA T4.

In [ ]:
# Install Unsloth and compatible libraries
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes
!pip install datasets huggingface_hub

### 2. Verify GPU & Hardware Profile
Ensure an NVIDIA T4 or P100 with at least 15 GB VRAM is active.

In [ ]:
import torch
assert torch.cuda.is_available(), "CUDA GPU is required for training! Please change runtime to T4 GPU."
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
print(f"Connected to: {gpu_name} with {vram_gb:.2f} GB VRAM")

### 3. Hugging Face Authentication (Secret / Environment Variable)
Reads `HF_TOKEN` from Colab Secrets or local environment. Never hardcode tokens in notebooks.

In [ ]:
import os
from huggingface_hub import login

hf_token = None
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    hf_token = os.environ.get('HF_TOKEN')

if hf_token:
    login(token=hf_token)
    print("Authenticated successfully with Hugging Face.")
else:
    print("Warning: HF_TOKEN not set. Set HF_TOKEN in Colab Secrets (key icon) or export HF_TOKEN to enable auto-checkpoint sync to Hub.")

### 4. Load Base Model in 4-bit with FastLanguageModel
We use `unsloth/Qwen2.5-7B-Instruct` (or `Qwen2.5-1.5B-Instruct` for ultra-fast runs) with `load_in_4bit=True`.

In [ ]:
from unsloth import FastLanguageModel

max_seq_length = 2048
dtype = None  # None for auto detection (fp16 on T4)
load_in_4bit = True
model_name = "unsloth/Qwen2.5-7B-Instruct"  # Core generalist foundation

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

### 5. Attach PEFT LoRA Adapters
Per `unsloth-t4-trainer` specs:
- `r = 16`, `lora_alpha = 32`
- Target all 7 linear projections: `q_proj`, `k_proj`, `v_proj`, `o_proj`, `gate_proj`, `up_proj`, `down_proj`
- `use_gradient_checkpointing = 'unsloth'`

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)
print("LoRA adapter attached successfully.")

### 6. Format Dataset into Qwen ChatML
Ensures messages retain `<think>...</think>` reasoning traces and format using the official Qwen chat template.

In [ ]:
from unsloth.chat_templates import get_chat_template
from datasets import load_dataset, Dataset

tokenizer = get_chat_template(
    tokenizer,
    chat_template="qwen-2.5",
)

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [
        tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False)
        for convo in convos
    ]
    return {"text": texts}

# Load local reasoning data if present, or initialize starter dataset
dataset_file = "data/reasoning_data.jsonl"
if os.path.exists(dataset_file):
    dataset = load_dataset("json", data_files=dataset_file, split="train")
else:
    print("Using embedded starter reasoning dataset with <think> traces...")
    starter_samples = [
        {
            "messages": [
                {"role": "system", "content": "You are QWEN-999B, a high-reasoning AI assistant specialized in logic, coding, and multilingual problem-solving."},
                {"role": "user", "content": "How many r's are in the word strawberry?"},
                {"role": "assistant", "content": "<think>\nAnalyze the word 'strawberry' letter by letter:\ns - 1\nt - 2\nr - 3 (first 'r')\na - 4\nw - 5\nb - 6\ne - 7\nr - 8 (second 'r')\nr - 9 (third 'r')\ny - 10\nTotal occurrences of 'r': 3.\n</think>\nThere are **3** \"r\"s in the word **strawberry**."}
            ]
        },
        {
            "messages": [
                {"role": "system", "content": "You are QWEN-999B, a high-reasoning AI assistant specialized in logic, coding, and multilingual problem-solving."},
                {"role": "user", "content": "Bhai explain karo what is Dijkstra's algorithm and iska time complexity kya hota hai?"},
                {"role": "assistant", "content": "<think>\nDijkstra's algorithm finds single-source shortest paths in weighted graphs with non-negative edge weights.\nUsing min-heap / priority queue:\nTime complexity: O((V + E) log V).\nUser asked in Hinglish, respond with friendly, precise technical Hinglish.\n</think>\nBhai **Dijkstra's Algorithm** ek greedy graph algorithm hai jo single source se graph ke baaki saare nodes ka **shortest path** nikalta hai.\n\n### Key Rules:\n1. **Non-negative weights:** Yeh algorithm sirf non-negative edge weights par kaam karta hai (negative weights ke liye Bellman-Ford use karte hain).\n2. **Min-Heap / Priority Queue:** Current minimum distance wale node ko explore karne ke liye min-heap use hota hai.\n\n### Time & Space Complexity:\n- **Time Complexity:** `O((V + E) log V)` (jab min-heap use karein, jahan V = vertices aur E = edges).\n- **Space Complexity:** `O(V)` distances aur visited tracking ke liye."}
            ]
        }
    ] * 50
    dataset = Dataset.from_list(starter_samples)

dataset = dataset.map(formatting_prompts_func, batched=True)
print(f"Dataset prepared with {len(dataset)} formatted ChatML samples.")

### 7. Configure SFTTrainer & TrainingArguments for T4
Strict adherence to T4 hardware guardrails:
- `per_device_train_batch_size = 2`
- `gradient_accumulation_steps = 4` (effective batch size 8)
- `fp16 = True`, `bf16 = False` (Mandatory for T4 stability)
- `optim = 'adamw_8bit'`
- `save_strategy = 'steps'`, `save_steps = 50`, `resume_from_checkpoint = True`

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

hub_model_id = "qwen-999b-adapter"  # Replace with your HF username/repo if pushing
output_dir = "outputs/qwen-999b-checkpoints"

training_args = TrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=10,
    max_steps=150,               # Set to num_train_epochs or steps as needed
    learning_rate=2e-4,
    fp16=True,                   # Required for NVIDIA T4
    bf16=False,                  # STRICT: T4 does not support BF16
    logging_steps=5,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="linear",
    seed=3407,
    save_strategy="steps",
    save_steps=50,               # Save every 50 steps against Colab timeouts
    save_total_limit=3,
    push_to_hub=bool(hf_token),  # Auto-push if token is present
    hub_model_id=hub_model_id if hf_token else None,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=training_args,
)

### 8. Train the Model
Resumes automatically from the latest checkpoint if interrupted.

In [ ]:
# Resume from checkpoint if interrupted
checkpoint_to_resume = True if os.path.isdir(output_dir) and any("checkpoint-" in d for d in os.listdir(output_dir)) else False
print(f"Resume from existing checkpoint: {checkpoint_to_resume}")

train_result = trainer.train(resume_from_checkpoint=checkpoint_to_resume)
print(f"Training finished with final loss: {train_result.training_loss:.4f}")

### 9. Save LoRA Adapters & Push to Hugging Face
Saves the trained QLoRA weights locally and uploads them to the Hugging Face Hub.

In [ ]:
final_lora_path = "outputs/qwen-999b-final-lora"
model.save_pretrained(final_lora_path)
tokenizer.save_pretrained(final_lora_path)
print(f"Saved final LoRA adapter to {final_lora_path}")

if hf_token:
    print("Pushing LoRA adapter to Hugging Face Hub...")
    model.push_to_hub(hub_model_id, token=hf_token)
    tokenizer.push_to_hub(hub_model_id, token=hf_token)
    print(f"Successfully synced to Hugging Face: {hub_model_id}")

### 10. Export to GGUF (Q4_K_M) for Ollama Inference
Per `gguf-ollama-packager` skill, export quantized GGUF directly for local Ryzen PC / Dell 15 Ollama execution.

In [ ]:
# Export to Q4_K_M GGUF format
gguf_output_dir = "outputs/qwen-999b-gguf"
print("Exporting model to Q4_K_M GGUF format for Ollama...")
try:
    model.save_pretrained_gguf(
        gguf_output_dir,
        tokenizer,
        quantization_method="q4_k_m"
    )
    print(f"GGUF export completed in {gguf_output_dir}!")
except Exception as e:
    print(f"GGUF export note: {e}")